# extract — prompt iteration

Calls the **node function directly**, not the graph. No Mongo or Qdrant writes —
just the LLM call, so the loop stays fast while you tune the prompt.

Needs `ollama serve` running.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print("tracing :", os.environ.get("LANGSMITH_TRACING"))
print("project :", os.environ.get("LANGSMITH_PROJECT"))

## 1. The input

In [ ]:
# Any real project description. Defaults to this repo's own README —
# it exists in a fresh clone, and it is exactly the kind of document
# this pipeline is built for.
RAW = open("../README.md").read()

# or point it at your own:
# RAW = open("../DESIGN.md").read()

print(len(RAW), "chars")

## 2. Run the node

Edit `src/uatu/prompts/extract.prompty`, then re-run this cell. `load` is cached,
so `load.cache_clear()` below is what makes your prompt edit take effect.

In [ ]:
from uatu.prompts import load
from uatu.graphs.ingest.node import extract

load.cache_clear()          # so edits to extract.prompty take effect

result = extract({"project_id": "notebook", "raw_text": RAW})
items = result["items"]

len(items)

## 3. Look at it

In [ ]:
for i in items:
    print(f"{i.label.value:<12} {len(i.text):>5}  {i.title}")

In [ ]:
from collections import Counter

print(Counter(i.label.value for i in items))
print("longest:", max(len(i.text) for i in items), "chars   (embedder truncates at ~1000)")

## 4. Did it invent anything?

Words in the output that never appeared in the input. The prompt says copy
verbatim, so this should print nothing.

In [ ]:
import re


def words(s):
    return set(re.findall(r"[a-zA-Z_][a-zA-Z0-9_]+", s.lower()))


source = words(RAW)

for i in items:
    invented = words(i.text) - source
    if invented:
        print(f"{i.title}: {sorted(invented)}")

## 5. Did it drop anything?

In [ ]:
total = sum(len(i.text) for i in items)
print(f"source {len(RAW)} chars -> extracted {total} chars ({total / len(RAW):.0%})")

## 6. The whole graph

Only once the table above looks right — this **does** write to Mongo and Qdrant,
so use a real project id from `POST /projects`.

In [ ]:
from uatu.graphs.ingest.graph import graph

for step in graph.stream({"project_id": "<real-project-id>", "raw_text": RAW}):
    print(step)